Alerta de ordens pendentes criação de KPI:<br>
ordens pendentes, quantidade de itens e informações de clientes para envio de alerta por email e telefone de quem tem o cadastro completo 

In [0]:
%python
bronze_path   = '/Volumes/bikestore/default/bikestore/bronze/'
silver_path   = '/Volumes/bikestore/default/bikestore/silver/'
gold_path     = '/Volumes/bikestore/default/bikestore/gold/'
resource_path = '/Volumes/bikestore/default/bikestore/resource/origem/'

In [0]:
%python
silver_map = {
    "tmp_silver_customer":      f"{silver_path}/customer/",
    "tmp_silver_orders":        f"{silver_path}/orders/",
    "tmp_silver_product":       f"{silver_path}/product/",

}
for view_name, path in silver_map.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name))
 


In [0]:
%python
df_orders_pending = spark.sql("""
                               
WITH pending AS(   

SELECT  
  customer_id
  ,order_date
  ,sum(quantity) quantity
  ,store_name
  --,status
  --,lower(status) as minusculo
  --,upper(status) as maiusculo
FROM tmp_silver_orders
WHERE 1=1
AND lower(status) = 'pending' 
--and lower(status) in ('pending','delivered') 
GROUP BY  customer_id,store_name,order_date
 --,status

)
SELECT 
  p.*
  ,c.first_name
  ,c.email
  ,c.phone

FROM pending  P
LEFT JOIN  tmp_silver_customer c ON P.customer_id = c.customer_id
WHERE c.email IS NOT NULL
AND c.phone IS NOT NULL
         
                              
                              """)

# salvar em Delta na gold
df_orders_pending.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema','true')\
    .save(f'{gold_path}/orders_pending')
    

In [0]:
%python
#CRIANDO A TABELA
df = df_orders_pending.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bikestore.logistics.gold_orders_pending")

In [0]:
select * from bikestore.logistics.gold_orders_pending

customer_id,order_date,quantity,store_name,first_name,email,phone
138,2018-04-27,4,Santa Cruz Bikes,Jone,jone.bernard@hotmail.com,(657) 536-5165
80,2018-04-22,5,Baldwin Bikes,Sarai,sarai.mckee@msn.com,(716) 912-8110
67,2018-04-17,1,Santa Cruz Bikes,Tommie,tommie.melton@gmail.com,(916) 802-2952
110,2018-04-13,3,Santa Cruz Bikes,Ollie,ollie.zimmerman@yahoo.com,(657) 648-2863
170,2018-04-17,4,Santa Cruz Bikes,Regine,regine.gonzales@gmail.com,(805) 763-4045
39,2018-04-17,8,Baldwin Bikes,Janetta,janetta.aguirre@aol.com,(717) 670-2634
56,2018-04-28,2,Rowlett Bikes,Lolita,lolita.mosley@hotmail.com,(281) 363-3309
5,2018-04-17,4,Santa Cruz Bikes,Charolette,charolette.rice@msn.com,(916) 381-6003
43,2018-04-29,8,Rowlett Bikes,Mozelle,mozelle.carter@aol.com,(281) 489-9656


In [0]:
--DROP TABLE bikestore.logistics.gold_orders_pending

In [0]:
--executar somente 1x
--create table if not exists bikestore.logistics.gold_orders_pending
--LOCATION 'abfss://uc-ext-azure@externalazure.dfs.core.windows.net/bikestore/gold/orders_pending'

In [0]:
--select * from bikestore.logistics.gold_orders_pending